# Phase 2: PostgreSQL CDC Validation

## Purpose
Validate that Change Data Capture (CDC) is working correctly:
1. PostgreSQL source database has data
2. Debezium is capturing changes
3. Changes are flowing to Kafka
4. Data can be queried from Kafka

## Setup
Before running this notebook:
1. Start all services: `docker-compose -f docker-compose.yml -f docker-compose-phase2.yml up -d`
2. Register connector: `./scripts/register-debezium-connector.sh`
3. Wait 30 seconds for initial snapshot

In [1]:
# Install required libraries
!pip install --quiet psycopg2-binary kafka-python

In [2]:
import psycopg2
import json
from kafka import KafkaConsumer
from datetime import datetime
import pandas as pd

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## Step 1: Connect to PostgreSQL Source Database

In [3]:
# Connect to HR system database
conn = psycopg2.connect(
    host="postgres-source-hr",
    port=5432,
    database="hr_system",
    user="hr_user",
    password="hr_pass123"
)
cursor = conn.cursor()

print("✅ Connected to PostgreSQL HR system")

✅ Connected to PostgreSQL HR system


## Step 2: Verify Source Data

In [4]:
# Check department counts
cursor.execute("SELECT COUNT(*) FROM hr.departments")
dept_count = cursor.fetchone()[0]
print(f"Departments: {dept_count}")

# Check employee counts
cursor.execute("SELECT COUNT(*) FROM hr.employees")
emp_count = cursor.fetchone()[0]
print(f"Employees: {emp_count}")

# Check job history counts
cursor.execute("SELECT COUNT(*) FROM hr.job_history")
history_count = cursor.fetchone()[0]
print(f"Job History Records: {history_count}")

print(f"\n✅ Source database has data")

Departments: 8
Employees: 21
Job History Records: 8

✅ Source database has data


In [5]:
# Show sample employees
query = """
SELECT 
    e.employee_id,
    e.first_name,
    e.last_name,
    e.email,
    e.job_title,
    d.department_name,
    e.salary
FROM hr.employees e
LEFT JOIN hr.departments d ON e.department_id = d.department_id
WHERE e.employment_status = 'active'
LIMIT 5
"""

df = pd.read_sql(query, conn)
print("\nSample Employees:")
df


Sample Employees:


/tmp/ipykernel_109/3824532286.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,employee_id,first_name,last_name,email,job_title,department_name,salary
0,1,Sarah,Chen,sarah.chen@company.com,VP of Engineering,Engineering,185000.0
1,6,Michael,Brown,michael.brown@company.com,VP of Product,Product,175000.0
2,9,Robert,Anderson,robert.anderson@company.com,VP of Sales,Sales,180000.0
3,12,Jessica,Garcia,jessica.garcia@company.com,VP of Marketing,Marketing,165000.0
4,14,Patricia,Moore,patricia.moore@company.com,VP of Human Resources,Human Resources,155000.0


## Step 3: Check Kafka Topics

In [6]:
from kafka import KafkaAdminClient

# Connect to Kafka
admin_client = KafkaAdminClient(
    bootstrap_servers='kafka:9092',
    client_id='validation'
)

# List all topics
topics = admin_client.list_topics()
hr_topics = [t for t in topics if t.startswith('hr')]

print("\nHR-related Kafka Topics:")
for topic in sorted(hr_topics):
    print(f"  • {topic}")

if hr_topics:
    print(f"\n✅ Found {len(hr_topics)} HR topics")
else:
    print("\n⚠️  No HR topics found. Did you register the connector?")

NoBrokersAvailable: NoBrokersAvailable

## Step 4: Read CDC Events from Kafka

In [ ]:
# Read latest messages from employees topic
consumer = KafkaConsumer(
    'hr.hr.employees',
    bootstrap_servers='kafka:9092',
    auto_offset_reset='earliest',
    enable_auto_commit=False,
    value_deserializer=lambda m: json.loads(m.decode('utf-8'))
)

print("Reading CDC events from Kafka...\n")

messages = []
for i, message in enumerate(consumer):
    messages.append(message.value)
    if i >= 4:  # Read first 5 messages
        break

consumer.close()

if messages:
    print(f"✅ Successfully read {len(messages)} CDC events")
    print("\nSample CDC Event:")
    print(json.dumps(messages[0], indent=2))
else:
    print("⚠️  No messages found. Connector may still be snapshotting.")

## Step 5: Test Real-Time CDC
Insert a new employee and watch it appear in Kafka

In [ ]:
# Insert a new employee
insert_query = """
INSERT INTO hr.employees 
(first_name, last_name, email, phone, hire_date, department_id, job_title, employment_status, employee_type, salary)
VALUES 
('Test', 'User', 'test.user@company.com', '555-555-9999', CURRENT_DATE, 1, 'Test Engineer', 'active', 'full_time', 100000.00)
RETURNING employee_id, first_name, last_name, email
"""

cursor.execute(insert_query)
new_employee = cursor.fetchone()
conn.commit()

print(f"✅ Inserted new employee:")
print(f"   ID: {new_employee[0]}")
print(f"   Name: {new_employee[1]} {new_employee[2]}")
print(f"   Email: {new_employee[3]}")
print(f"\n⏳ Waiting for CDC event...")

In [ ]:
import time

# Listen for the new employee in Kafka
consumer = KafkaConsumer(
    'hr.hr.employees',
    bootstrap_servers='kafka:9092',
    auto_offset_reset='latest',  # Only read new messages
    enable_auto_commit=False,
    consumer_timeout_ms=10000,  # Wait max 10 seconds
    value_deserializer=lambda m: json.loads(m.decode('utf-8'))
)

found_event = None
for message in consumer:
    event = message.value
    if event.get('email') == 'test.user@company.com':
        found_event = event
        break

consumer.close()

if found_event:
    print("✅ CDC event detected in Kafka!\n")
    print("Event details:")
    print(json.dumps(found_event, indent=2, default=str))
else:
    print("⚠️  CDC event not detected within timeout. Check Debezium logs.")

## Step 6: Test Update CDC Event

In [ ]:
# Update the test employee's salary
update_query = """
UPDATE hr.employees 
SET salary = 110000.00, job_title = 'Senior Test Engineer'
WHERE email = 'test.user@company.com'
RETURNING employee_id, first_name, last_name, salary, job_title
"""

cursor.execute(update_query)
updated_employee = cursor.fetchone()
conn.commit()

print(f"✅ Updated employee:")
print(f"   ID: {updated_employee[0]}")
print(f"   Name: {updated_employee[1]} {updated_employee[2]}")
print(f"   New Salary: ${updated_employee[3]:,.2f}")
print(f"   New Title: {updated_employee[4]}")
print(f"\n⏳ Waiting for UPDATE CDC event...")

In [ ]:
# Listen for the update event
consumer = KafkaConsumer(
    'hr.hr.employees',
    bootstrap_servers='kafka:9092',
    auto_offset_reset='latest',
    enable_auto_commit=False,
    consumer_timeout_ms=10000,
    value_deserializer=lambda m: json.loads(m.decode('utf-8'))
)

update_event = None
for message in consumer:
    event = message.value
    if event.get('email') == 'test.user@company.com' and event.get('salary') == 110000.0:
        update_event = event
        break

consumer.close()

if update_event:
    print("✅ UPDATE CDC event detected!\n")
    print(f"New salary: ${update_event['salary']:,.2f}")
    print(f"New title: {update_event['job_title']}")
else:
    print("⚠️  UPDATE event not detected")

## Step 7: Cleanup Test Data

In [ ]:
# Delete the test employee
cursor.execute("DELETE FROM hr.employees WHERE email = 'test.user@company.com'")
conn.commit()
print("✅ Test employee deleted")

# Close connection
cursor.close()
conn.close()
print("✅ PostgreSQL connection closed")

## Summary

### ✅ Phase 2 CDC Validation Complete!

We validated:
1. PostgreSQL source database is populated with HR data
2. Debezium is capturing changes via logical replication
3. CDC events are flowing to Kafka topics
4. INSERT, UPDATE operations are captured in real-time
5. Events contain full record data in JSON format

### Next Steps:

**Phase 2b: Write to Iceberg**
- Create Kafka Connect sink connector
- Write CDC events to Iceberg raw layer
- Validate data in Iceberg tables

**Phase 3: Orchestration**
- Add Apache Airflow
- Schedule data pipelines
- Monitor CDC health

### Useful Links:
- Kafka UI: http://localhost:8090
- Debezium API: http://localhost:8083
- PostgreSQL Source: localhost:5433 (user: hr_user, db: hr_system)